# SixtyFour AI — Fast Company Search & Decision Makers

This notebook uses the Sixtyfour **`/search/query`** endpoint to resolve a company from a **website and/or name**, then pull its **finance & operations decision makers** — in seconds, not minutes.

### How this differs from `enrich-company`

| | `enrich-company` | `/search/query` (this notebook) |
|---|---|---|
| **How it works** | An agent reads the live web | Queries a pre-built index |
| **Latency** | 5–10 minutes | **~5–10 seconds** |
| **Best for** | Open-ended questions, any field you can describe | Firmographics + people already in the index |
| **Returns** | Exactly the `struct` you asked for | A fixed, structured company profile |

Reach for this one when you need **speed and volume** — scoring an inbound list, hydrating a CRM, qualifying leads in a form handler. Reach for [`enrich-company`](sixtyfour_enrich_company_tutorial.ipynb) when you need a field no index carries.

### Why it returns a *list*

One domain or name can legitimately match several companies: LinkedIn showcase pages, regional subsidiaries, acquisitions, and unrelated firms that share a name. Searching `ramp.com` surfaces Ramp, Ramp Health, Ramp Network, and American Ramp Company.

So this **always returns a list**, best match first, each with a `match_score` — rather than silently guessing which one you meant.

## Prerequisites & Secrets

### Get your SixtyFour API key

**1) Sign in** — go to https://app.sixtyfour.ai/login and log in or sign up.

**2)** Enter an organization name, click **Create Organization**, and get your $25 in credits.

**3)** On the dashboard, open **Keys** in the left sidebar → **Create new key** → name it → **Create key** → copy the key value.

### If you're in Google Colab
- Click the 🔑 (Secrets) icon in the left sidebar.
- Add a secret named `SIXTYFOUR_API_KEY` and paste your key.
- Toggle **Notebook access** on for that secret.

### If you're running locally
```bash
export SIXTYFOUR_API_KEY="your_key_here"
jupyter notebook
```

> ⚠️ **Never paste your key into a notebook cell.** Anything typed into a cell gets committed to git and shared with the notebook. The cell below reads it from Colab Secrets or an environment variable, and never prints it.

In [1]:
# --- Parameters (you can change these) ---
COMPANY_WEBSITE = "https://www.airwallex.com"
COMPANY_NAME    = "Airwallex"

import os

API_KEY = os.getenv("SIXTYFOUR_API_KEY", "")   # local / already-set env var
try:
    from google.colab import userdata           # Colab Secrets
    secret_val = userdata.get("SIXTYFOUR_API_KEY")
    if secret_val:
        os.environ["SIXTYFOUR_API_KEY"] = secret_val
        API_KEY = secret_val
except Exception:
    pass   # not in Colab, or the secret was not granted

if not API_KEY:
    raise RuntimeError(
        "No API key found. In Colab: add a Secret named SIXTYFOUR_API_KEY. "
        "Locally: export SIXTYFOUR_API_KEY=... before starting Jupyter."
    )

print("✅ Setup complete (key detected, endpoint ready).")

✅ Setup complete (key detected, endpoint ready).


## How it works

Two stages, each fanned out across a thread pool:

**Stage 1 — resolve the company.** Two searches fire at once: an exact `website_domain` lookup and a fuzzy `company_name` match. Results are merged, deduped, scored, and ranked.

**Stage 2 — fetch the people.** For every surviving company, one query per seniority bucket fires *in parallel* (`CXO`, `VP`, `Director`, …). This is the important trick: the obvious implementation asks for all seniorities in one query and walks the cursor page by page, which is **sequential**. Splitting by seniority turns that into six concurrent queries that each usually fit in a single page.

Nothing is nested — every request is submitted to one flat pool — so no task ever blocks waiting on a slot another task is holding.

### Two details that matter more than they look

- **Connection pool size.** `urllib3` pools only 10 connections by default. Left alone, concurrent requests silently queue behind each other and the fan-out buys you nothing. The session below sizes its pool to the thread pool.
- **Filter people on `companyId`, not `company.url`.** For Airwallex, `companyId` matches 2,469 people; `company.url` matches 238. Using the URL costs ~90% of the roster.

In [2]:
import re, time, difflib, json
from concurrent.futures import ThreadPoolExecutor
from contextlib import closing

import requests
from requests.adapters import HTTPAdapter

API_URL = "https://api.sixtyfour.ai/search/query"

# Leadership/Entrepreneurship matter: without them you lose every CEO and
# founder, whose jobFunction is never "Finance".
JOB_FUNCTIONS = ["Finance", "Accounting", "Operations", "Leadership", "Entrepreneurship"]
SENIORITY = ["CXO", "VP", "Director", "Head", "Owner / Founder", "Partner"]
HEADLINE_KEYWORDS = "finance financial controller treasury accounting operations"

# Stripped before comparing names: they carry no identifying signal, and
# leaving them in scores "Acme Corporation" against "ZTE Corporation" at 0.84.
GENERIC_NAME_TOKENS = {
    "inc", "llc", "ltd", "limited", "corp", "corporation", "co", "company",
    "group", "holdings", "holding", "plc", "gmbh", "ag", "bv", "nv", "pty",
    "sa", "srl", "spa", "kk", "the", "of", "and", "member", "international",
}

# Ranked ICP titles. Tier 1 = the decision maker, 4 = worth having.
ICP = [
    (1, r"\b(chief executive officer|ceo)\b"),
    (1, r"\b(chief financial officer|cfo)\b"),
    (1, r"\b(chief operating officer|coo)\b"),
    (1, r"\b(chief revenue officer|cro)\b"),
    (1, r"\b(co[- ]?founder|founder)\b"),
    (2, r"\bchief\s+[a-z]+\s+officer\b"),
    (2, r"\b(vp|vice president|svp|evp|head)\s+(of\s+)?(finance|operations)\b"),
    (3, r"\bdirector\s+(of\s+)?(finance|operations)\b"),
    (3, r"\b(financial\s+)?controller\b"),
    (4, r"\b(treasurer|treasury director|chief accounting officer)\b"),
]

print("✅ Config loaded.")

✅ Config loaded.


### Transport: pooling and retries

The index allows **200 requests per minute**. A wide fan-out can brush that ceiling, and a swallowed `429` is indistinguishable from *"this company has no decision makers"* — the worst possible failure, because it looks like a valid answer.

So: retries honour the API's own `Retry-After` hint, and anything finally abandoned gets **recorded** rather than hidden.

In [3]:
def _session(api_key, pool_size):
    """Session whose connection pool is big enough for the thread pool.

    urllib3 defaults to 10 pooled connections; without this, concurrent
    requests queue up behind each other and the fan-out buys nothing.
    """
    s = requests.Session()
    s.headers.update({"x-api-key": api_key})
    adapter = HTTPAdapter(pool_connections=pool_size, pool_maxsize=pool_size)
    s.mount("https://", adapter)
    s.mount("http://", adapter)
    return s


def _retry_after(response, fallback):
    """Seconds to wait, from the Retry-After header or the JSON detail."""
    header = response.headers.get("Retry-After")
    if header:
        try:
            return float(header)
        except ValueError:
            pass
    m = re.search(r"retry after (\d+(?:\.\d+)?) second", response.text or "", re.I)
    return float(m.group(1)) + 0.5 if m else fallback


def _make_post(session, timeout, retries, errors):
    """POST that retries 429/5xx and records, rather than hides, failures."""
    def post(body):
        delay = 1.0
        for attempt in range(retries + 1):
            try:
                r = session.post(API_URL, json=body, timeout=timeout)
            except requests.RequestException as e:
                if attempt == retries:
                    errors.append(f"{type(e).__name__}: {e}")
                    return None
                time.sleep(delay)
                delay = min(delay * 2, 30)
                continue

            if r.status_code == 200:
                return r.json()
            if r.status_code == 429 or r.status_code >= 500:
                if attempt == retries:
                    errors.append(f"HTTP {r.status_code}: {r.text[:200]}")
                    return None
                time.sleep(min(_retry_after(r, delay), 30))
                delay = min(delay * 2, 30)
                continue
            errors.append(f"HTTP {r.status_code}: {r.text[:200]}")
            return None
        return None
    return post

print("✅ Transport ready.")

✅ Transport ready.


### Matching: which company did you actually mean?

`match_score` runs 0–1: the domain is worth 60%, the name 40%.

The name half is the subtle part. Comparing raw strings scores `"Acme Corporation"` against `"ZTE Corporation"` at **0.84** — almost entirely on the shared word *Corporation*. So generic legal tokens are stripped first, and a match then needs either a **shared whole token** or a **near-identical string** (so typos like `"Airwalex"` still resolve). Junk falls to ~0.001 and gets filtered by `min_score`.

In [4]:
def _unwrap(v):
    """The index wraps every list item as {"element": X}. Strip that."""
    if isinstance(v, list):
        return [_unwrap(x) for x in v]
    if isinstance(v, dict):
        return _unwrap(v["element"]) if set(v) == {"element"} else {k: _unwrap(x) for k, x in v.items()}
    return v


def _domain(url):
    """'https://www.airwallex.com/us/about' -> 'airwallex.com'."""
    if not url:
        return None
    host = re.sub(r"^[a-z]+://", "", str(url).strip(), flags=re.I).split("/")[0]
    host = re.sub(r"^www\.", "", host, flags=re.I).split(":")[0].lower()
    if not host:
        return None
    parts = host.split(".")
    # keep three labels for co.uk / com.au style suffixes, two otherwise
    if len(parts) > 2 and parts[-2] in {"co", "com", "net", "org", "gov", "ac"}:
        return ".".join(parts[-3:])
    return ".".join(parts[-2:]) if len(parts) > 1 else host


def _norm(s):
    return " ".join(re.findall(r"[a-z0-9&]+", (s or "").lower()))


def _distinctive(s):
    """Normalized name minus legal/filler tokens, for like-for-like compare."""
    tokens = [t for t in _norm(s).split() if t not in GENERIC_NAME_TOKENS]
    return " ".join(tokens) or _norm(s)          # fall back if nothing is left


def _as_list(v):
    v = _unwrap(v)
    if v is None:
        return []
    items = v if isinstance(v, list) else [v]
    return [x for x in items if isinstance(x, str)]


def _score(src, domain, name):
    """0..1 confidence that `src` is the company the caller meant."""
    parts, weights = [], []

    if domain:
        cand = _domain(src.get("website_domain") or src.get("website"))
        parts.append(1.0 if cand == domain else 0.0)
        weights.append(0.6)

    if name:
        want = _distinctive(name)
        want_tokens = set(want.split())
        best = 0.0
        for field in ("company_name", "company_legal_name", "company_name_alias"):
            for value in _as_list(src.get(field)):
                got = _distinctive(value)
                if not got:
                    continue
                got_tokens = set(got.split())
                ratio = difflib.SequenceMatcher(None, want, got).ratio()
                # Character overlap alone is noise on short names ("acme" vs
                # "danaher" scores .36). Demand a shared whole token, or a
                # near-identical string so typos still resolve.
                if want_tokens & got_tokens:
                    # "Airwallex" inside "MexPago | Member of the Airwallex
                    # Group" is real, but weaker than an exact name.
                    if want_tokens <= got_tokens:
                        ratio = max(ratio, 0.75)
                elif ratio < 0.7:
                    ratio = 0.0
                best = max(best, ratio)
        parts.append(best)
        weights.append(0.4)

    total = sum(weights) or 1.0
    base = sum(p * w for p, w in zip(parts, weights)) / total
    # nudge real companies above 0-employee showcase pages at equal scores
    return base + (0.001 if (src.get("employees_count") or 0) > 0 else 0.0)

print("✅ Matching ready.")

✅ Matching ready.


### People: sharded fetch, then title confirmation

`seniority`, `jobFunction`, and `headline` are **person-level** fields — they fire if *any* current employer matches. So after fetching, each person's title is re-confirmed against *this* company's entry before they count.

Two query quirks worth knowing:
- `$match` on `currentCompanies.positions.title` **times out**, even on a single token. Keyword search goes against `headline`, a fast top-level field.
- `jobFunction` and `headline` must be `$or`'d. Chaining them with `$and` drops every CEO and COO.

In [5]:
def _people_shard(post, company, seniority, max_results):
    """All people at `company` in one seniority bucket, following cursors.

    Returns (rows, complete). `complete` is False when a request was given
    up on, so an empty list is never mistaken for a confirmed empty roster.
    """
    body = {
        "mode": "people",
        "simple_filters": {"$and": [
            {"currentCompanies": {"$elemMatch": {
                "currentCompanies.company.companyId": int(company["source_id"])
            }}},
            {"seniority": seniority},
            {"$or": [
                {"jobFunction": {"$in": JOB_FUNCTIONS}},
                {"headline": {"$match": HEADLINE_KEYWORDS}},
            ]},
        ]},
        "page_size": 100,
        "max_results": max_results,
        "output_shape": "raw",
    }

    rows, page = [], post(body)
    while True:
        if page is None:
            return rows, False                   # gave up: not "empty"
        for row in page.get("results") or []:
            rows.append(row.get("raw_source") or row)
            if len(rows) >= max_results:
                return rows, True
        cursor = page.get("next_cursor")
        if not cursor or not page.get("has_more"):
            return rows, True
        page = post({"cursor": cursor})           # later pages send ONLY the cursor


def _icp_tier(title):
    """Lowest ICP tier this title matches, or None if it is off-ICP."""
    t = re.sub(r"[^a-z0-9 ]+", " ", (title or "").lower())
    return next((n for n, p in ICP if re.search(p, t)), None)


def _rank_contacts(people, company, min_tier):
    """Dedupe, confirm each title at this company, keep on-ICP, sort."""
    company_id = int(company["source_id"])
    company_name = company.get("company_name")
    out, seen = [], set()

    for p in people:
        pid = p.get("publicId")
        if not pid or pid in seen:
            continue

        # seniority/jobFunction/headline are PERSON-level: they fire on any
        # current employer. Confirm the title against this company's entry.
        for cc in p.get("currentCompanies") or []:
            if (cc.get("company") or {}).get("companyId") != company_id:
                continue

            # positions[] is absent on some profiles. Employment is already
            # proven by the companyId match, so the headline only has to
            # supply the title.
            cands = [(pos.get("title"), "position") for pos in (cc.get("positions") or [])]
            if not cands:
                cands = [(p.get("headline"), "headline")]

            hit = next(((t, s, _icp_tier(t)) for t, s in cands if _icp_tier(t)), None)
            if hit and hit[2] <= min_tier:
                title, source, tier = hit
                seen.add(pid)
                out.append({
                    "name": p.get("fullName"),
                    "title": title,
                    "title_source": source,      # "headline" = weaker evidence
                    "icp_tier": tier,
                    "linkedin_url": f"https://www.linkedin.com/in/{pid}",
                    "location": p.get("locationName"),
                    "seniority": p.get("seniority"),
                    "job_function": p.get("jobFunction"),
                    "company_name": company_name,
                    "company_id": company_id,
                    "company_confirmed": True,
                    "email": None,               # not in this index
                    "phone": None,               # not in this index
                })
            break

    out.sort(key=lambda c: (c["icp_tier"], c["title_source"] == "headline"))
    return out

print("✅ People logic ready.")

✅ People logic ready.


### Flattening a company row

The index returns hundreds of raw fields. `_clean_company` reduces one row to a flat profile: identity, location, size, revenue, and international footprint.

In [6]:
def _clean_company(src):
    """Sixtyfour /search/query company row -> flat company profile dict."""

    def first(*vals):
        return next((v for v in vals if v not in (None, "", [], {})), None)

    src = _unwrap(src)
    domain = _domain(src.get("website_domain") or src.get("website"))
    hq_country = _norm(src.get("hq_country"))

    # headcount by country, biggest first; anything outside HQ is international
    by_country = sorted(
        ({"country": r.get("country"), "employees": r.get("employee_count")}
         for r in src.get("employees_count_by_country") or [] if isinstance(r, dict) and r.get("country")),
        key=lambda r: r["employees"] or 0, reverse=True)
    intl_countries = [r["country"] for r in by_country if _norm(r["country"]) != hq_country]

    regions = {k.replace("employees_count_", "").replace("_", " "): v
               for k, v in (src.get("employees_count_breakdown_by_region") or {}).items()
               if v and k != "employees_count_unknown"}

    offices = [{"address": o.get("location_address"), "city": o.get("city"),
                "state": o.get("state"), "is_primary": bool(o.get("is_primary"))}
               for o in src.get("company_locations_full") or [] if isinstance(o, dict)]

    emails = [e for e in src.get("company_emails") or [] if isinstance(e, str)]

    # revenue lands either as a bare number or as one block per data provider
    rev = rev_ccy = None
    node = src.get("revenue_annual")
    if isinstance(node, (int, float)):
        rev = node
    elif isinstance(node, dict):
        for b in node.values():
            if isinstance(b, dict) and b.get("annual_revenue") is not None:
                rev, rev_ccy = b["annual_revenue"], b.get("annual_revenue_currency")
                break

    fund = src.get("last_funding_round") or {}
    growth = src.get("employees_count_change") or {}

    return {
        # --- identity ---
        "company_name": first(src.get("company_name"), src.get("company_legal_name")),
        "company_legal_name": src.get("company_legal_name"),
        "company_id": first(src.get("source_id"), src.get("company_id")),
        "website": first(src.get("website"), src.get("website_domain")),
        "website_domain": domain,
        "linkedin_url": first(src.get("canonical_linkedin_url"), src.get("linkedin_url")),
        "description": first(src.get("description_enriched"), src.get("description")),
        "industry": src.get("industry"),
        "founded_year": src.get("founded_year"),
        # --- location ---
        "hq": first(src.get("hq_full_address"), src.get("hq_location")),
        "hq_city": src.get("hq_city"),
        "hq_state": src.get("hq_state"),
        "hq_country": src.get("hq_country"),
        "offices": offices,
        "office_count": len(offices),
        # --- size ---
        "employees_count": first(src.get("employees_count"), src.get("employees_count_inferred")),
        "employees_count_is_inferred": src.get("employees_count") is None,
        "size_range": src.get("size_range"),
        "employees_growth_yoy_pct": growth.get("change_yearly_percentage"),
        "employees_by_department": {k.replace("employees_count_", ""): v for k, v
                                    in (src.get("employees_count_breakdown_by_department") or {}).items() if v},
        # --- money ---
        "annual_revenue": rev,
        "annual_revenue_currency": rev_ccy,
        "annual_revenue_range": src.get("revenue_annual_range"),
        "is_public": bool(src.get("is_public")),
        "last_funding_type": fund.get("type"),
        "last_funding_amount": fund.get("amount_raised"),
        "last_funding_date": fund.get("announced_date"),
        # --- international footprint ---
        "employees_by_country": by_country,
        "international_employee_countries": intl_countries,
        "employees_by_region": regions,
        "has_international_presence": bool(intl_countries) or len(regions) > 1,
        # --- company-level contact info (people come from the people index) ---
        "company_emails": emails,
        "company_emails_on_domain": [e for e in emails if domain and e.lower().split("@")[-1].endswith(domain)],
        "company_phone_numbers": src.get("company_phone_numbers") or [],
    }

print("✅ Cleaning ready.")

✅ Cleaning ready.


## The function

Everything above, wired together. This is the only function you call.

In [7]:
def enrich_company(
    url=None,
    name=None,
    api_key=None,
    max_companies=10,
    max_contacts=200,
    min_tier=4,
    min_score=0.3,
    max_workers=32,
    timeout=90,
    retries=4,
):
    """Resolve a company from a website and/or name, with decision makers.

    A domain or a name can legitimately match several companies (subsidiaries,
    LinkedIn showcase pages, acquisitions), so this always returns a list --
    best match first, up to `max_companies`.

    Args:
        url: company website, e.g. "https://www.airwallex.com". Optional if
            `name` is given.
        name: company name, e.g. "Airwallex". Optional if `url` is given.
        api_key: Sixtyfour key. Defaults to $SIXTYFOUR_API_KEY.
        max_companies: cap on returned companies (default 10).
        max_contacts: cap on people fetched per seniority shard, per company.
        min_tier: keep contacts at this ICP tier or better (1 = strictest).
        min_score: drop companies scoring below this. Guards against the
            index returning loosely-related rows for an unknown company.
        max_workers: size of the shared thread/connection pool.
        timeout: per-request timeout in seconds.
        retries: attempts after a 429/5xx before giving up on one request.

    Returns:
        List of flat company dicts, best match first, each with three added
        keys: "match_score" (0-1 confidence this is the company you meant),
        "decision_makers" (ICP-ranked, best tier first), and
        "decision_makers_complete" (False if a lookup was abandoned, so an
        empty roster is not mistaken for a confirmed-empty one). Any
        abandoned request is also listed under "_errors" on the first
        company. Returns [] when nothing matches.
    """
    if not url and not name:
        raise ValueError("need url or name")

    api_key = api_key or os.environ.get("SIXTYFOUR_API_KEY")
    if not api_key:
        raise ValueError("no API key: pass api_key= or set $SIXTYFOUR_API_KEY")

    domain = _domain(url)
    errors = []

    with closing(_session(api_key, max_workers)) as session, \
            ThreadPoolExecutor(max_workers=max_workers) as pool:
        post = _make_post(session, timeout, retries, errors)

        # ---- stage 1: both company searches at once -----------------------
        searches = []
        if domain:
            searches.append({"website_domain": domain})
        if name:
            searches.append({"company_name": {"$match": name}})

        found = pool.map(
            lambda flt: post({
                "mode": "company",
                "simple_filters": {**flt, "$sort": {"employees_count": -1}},
                "page_size": max_companies * 2,
                "max_results": max_companies * 2,
                "output_shape": "raw",
            }),
            searches,
        )

        # dedupe across both searches, rank, truncate
        by_id = {}
        for page in found:
            for row in (page or {}).get("results") or []:
                src = row.get("raw_source") or row
                key = str(src.get("source_id") or src.get("company_id") or id(src))
                by_id.setdefault(key, src)

        scored = ({"src": s, "score": _score(s, domain, name)} for s in by_id.values())
        ranked = sorted(
            (c for c in scored if c["score"] >= min_score),
            key=lambda c: (-c["score"], -(c["src"].get("employees_count") or 0)),
        )[:max_companies]
        if not ranked:
            # "nothing matched" and "the lookup failed" must not look alike
            if errors:
                raise RuntimeError(f"company search failed: {errors[0]}")
            return []

        # ---- stage 2: every (company x seniority) query at once ------------
        # Sharding by seniority replaces sequential cursor paging with one
        # parallel query per bucket, which is where most of the wall clock
        # went before.
        tasks = [
            (i, c["src"], sen)
            for i, c in enumerate(ranked)
            for sen in SENIORITY
            # a 0-employee row is a LinkedIn showcase page; it has no roster,
            # so querying it only burns rate limit the real companies need
            if c["src"].get("source_id") and c["src"].get("employees_count") != 0
        ]
        shards = list(pool.map(
            lambda t: (t[0],) + _people_shard(post, t[1], t[2], max_contacts),
            tasks,
        ))

    # ---- assemble -------------------------------------------------------
    rows_by_company, complete = {}, {}
    for idx, rows, ok in shards:
        rows_by_company.setdefault(idx, []).extend(rows)
        complete[idx] = complete.get(idx, True) and ok

    out = []
    for i, c in enumerate(ranked):
        company = _clean_company(c["src"])
        company["match_score"] = round(c["score"], 4)
        company["decision_makers"] = _rank_contacts(
            rows_by_company.get(i, []), c["src"], min_tier
        )
        # False means a lookup was given up on, so an empty list above is
        # "we could not finish", not "this company has nobody".
        company["decision_makers_complete"] = complete.get(i, True)
        out.append(company)
    if errors:
        out[0]["_errors"] = errors
    return out

print("✅ enrich_company ready.")

✅ enrich_company ready.


## Run it

Uses `COMPANY_WEBSITE` and `COMPANY_NAME` from the parameters cell at the top.

In [8]:
start = time.time()
companies = enrich_company(url=COMPANY_WEBSITE, name=COMPANY_NAME)
elapsed = time.time() - start

print(f"{len(companies)} matching companies in {elapsed:.1f}s\n")
print(f"{'score':>6}  {'company':<42} {'domain':<22} {'employees':>9}  {'contacts':>8}")
print("-" * 95)
for c in companies:
    flag = "" if c["decision_makers_complete"] else "  ⚠️ incomplete"
    print(f"{c['match_score']:>6.3f}  {str(c['company_name'])[:42]:<42} "
          f"{str(c['website_domain']):<22} {str(c['employees_count']):>9}  "
          f"{len(c['decision_makers']):>8}{flag}")

4 matching companies in 6.7s

 score  company                                    domain                 employees  contacts
-----------------------------------------------------------------------------------------------
 1.001  Airwallex                                  airwallex.com               2625        18
 0.900  Airwallex Customer Updates                 airwallex.com                  0         0
 0.301  MexPago | Member of the Airwallex Group    mexpago.com                   14         0
 0.301  OpenPay (acq Airwallex)                    getopenpay.com                 3         0


### Reading `match_score`

When you pass **both** a URL and a name, the domain anchors the score and the tiers separate cleanly:

| Score | Means | Example for `airwallex.com` + `"Airwallex"` |
|---|---|---|
| **~1.0** | Domain **and** name both match | `Airwallex` |
| **~0.9** | Domain matches, name differs | `Airwallex Customer Updates` (a showcase page) |
| **~0.3–0.6** | Name relates, different domain | `MexPago \| Member of the Airwallex Group` |
| **< 0.3** | Filtered out by `min_score` | unrelated firms sharing a word |

A single confident answer is `companies[0]` when `match_score > 0.9`. Below that, the honest move is to show the list and let a human pick.

In [9]:
top = companies[0]

print(f"{top['company_name']}  —  {top['website']}")
print(f"{top['industry']}  |  founded {top['founded_year']}  |  {top['employees_count']} employees")
hq = ", ".join(x for x in (top["hq_city"], top["hq_state"], top["hq_country"]) if x)
print(f"HQ: {hq or 'n/a'}   ({top['office_count']} offices)")
print(f"LinkedIn: {top['linkedin_url']}")
if top["international_employee_countries"]:
    print(f"International: {len(top['international_employee_countries'])} countries beyond HQ")

print(f"\nDecision makers ({len(top['decision_makers'])}):\n")
print(f"{'tier':<5} {'name':<26} {'title':<44} {'evidence'}")
print("-" * 92)
for d in top["decision_makers"]:
    print(f"{d['icp_tier']:<5} {str(d['name'])[:26]:<26} {str(d['title'])[:44]:<44} {d['title_source']}")

Airwallex  —  https://www.airwallex.com
Financial Services  |  founded 2015  |  2625 employees
HQ: United States   (17 offices)
LinkedIn: https://www.linkedin.com/company/airwallex
International: 24 countries beyond HQ

Decision makers (18):

tier  name                       title                                        evidence
--------------------------------------------------------------------------------------------
1     JACK ZHANG                 Chief Executive Officer                      position
1     Lucy Yueting Liu           Co-founder & President                       position
1     Jacob Dai                  Co-Founder / CTO                             position
1     Jack Zhang                 Co-Founder & CEO                             position
1     Max L.                     Co-Founder                                   position
1     Pranav Sood                Chief Financial Officer                      position
2     Justin Yek                 VP, Finance           

### ICP tiers

Contacts are ranked by how close the title is to the buying decision:

| Tier | Titles |
|---|---|
| **1** | CEO, CFO, COO, CRO, Founder / Co-founder |
| **2** | Any other C-level; VP/SVP/EVP/Head of Finance or Operations |
| **3** | Director of Finance/Operations; Controller |
| **4** | Treasurer, Treasury Director, Chief Accounting Officer |

`title_source` is your evidence grade: **`position`** means the title came from a confirmed position record at this company; **`headline`** means it came from the person's LinkedIn headline, which is weaker. Employment is proven either way — the headline only supplies the wording.

Pass `min_tier=2` to keep only tiers 1–2.

## Variations

### Name only — and why you should pass a URL when you have one

With no domain to anchor the match, the score is computed **entirely** from the name — so every company that genuinely carries that name ties at the top, and the score cannot separate them. Note below that four different companies score `1.001`: each really is named "Ramp" (the index matches `company_name_alias` too). Only headcount breaks the tie.

This is honest behaviour rather than a bug — but it is the reason to pass `url` whenever you have one. **Ambiguity that the data cannot resolve should surface as a tie, not as a confident wrong answer.**

In [10]:
for c in enrich_company(name="Ramp")[:6]:
    print(f"{c['match_score']:.3f}  {str(c['company_name'])[:38]:<38} "
          f"{str(c['website_domain']):<24} dm={len(c['decision_makers'])}")

1.001  Ramp                                   ramp.com                 dm=8
1.001  Ramp Technology Group                  rampgroup.com            dm=1
1.001  The Ramp                               theramp.org              dm=0
1.001  The Ramp | Groupe EDG                  theramp.co               dm=1
0.751  Ramp Health                            ramphealth.com           dm=5
0.751  Ramp Network                           rampnetwork.com          dm=5


### Only the top of the house (`min_tier=2`), one company (`max_companies=1`)

In [11]:
exec_only = enrich_company(
    url=COMPANY_WEBSITE, name=COMPANY_NAME,
    min_tier=2,          # tiers 1-2 only
    max_companies=1,     # skip the showcase pages and affiliates
)
for d in exec_only[0]["decision_makers"]:
    print(f"t{d['icp_tier']}  {str(d['name'])[:26]:<26} {d['title']}")

t1  JACK ZHANG                 Chief Executive Officer
t1  Lucy Yueting Liu           Co-founder & President
t1  Jacob Dai                  Co-Founder / CTO
t1  Jack Zhang                 Co-Founder & CEO
t1  Max L.                     Co-Founder
t1  Pranav Sood                Chief Financial Officer
t2  Justin Yek                 VP, Finance
t2  Ge Fu                      VP, Operations, GC & SEA
t2  Aaron Sonenfeld            Head of Operations, EMEA
t2  Laurence Cua               Head of Operations ANZ


### Nothing matched vs. the lookup failed

These must never look alike. An unknown company returns an empty list; a failed lookup raises.

In [12]:
print("unknown company ->", enrich_company(
    url="https://not-a-real-domain-xyz123.com", name="Nonexistent Zzz Co"))

try:
    enrich_company()
except ValueError as e:
    print("no arguments     -> ValueError:", e)

unknown company -> []
no arguments     -> ValueError: need url or name


### Export to CSV — one row per contact

In [13]:
import csv, io

rows = [
    {**d, "company_domain": c["website_domain"], "company_match_score": c["match_score"]}
    for c in companies for d in c["decision_makers"]
]

if rows:
    buf = io.StringIO()
    w = csv.DictWriter(buf, fieldnames=list(rows[0]))
    w.writeheader()
    w.writerows(rows)
    # to save locally instead:  open("contacts.csv", "w").write(buf.getvalue())
    print(f"{len(rows)} contact rows\n")
    print("\n".join(buf.getvalue().splitlines()[:4]))

18 contact rows

name,title,title_source,icp_tier,linkedin_url,location,seniority,job_function,company_name,company_id,company_confirmed,email,phone,company_domain,company_match_score
JACK ZHANG,Chief Executive Officer,position,1,https://www.linkedin.com/in/jack-zhang-54a204294,"Ikeja, Lagos State, Nigeria",CXO,Leadership,Airwallex,7572649,True,,,airwallex.com,1.001
Lucy Yueting Liu,Co-founder & President,position,1,https://www.linkedin.com/in/awxlucy,Singapore,Owner / Founder,Entrepreneurship,Airwallex,7572649,True,,,airwallex.com,1.001
Jacob Dai,Co-Founder / CTO,position,1,https://www.linkedin.com/in/jacob-dai-615956110,Singapore,Owner / Founder,Entrepreneurship,Airwallex,7572649,True,,,airwallex.com,1.001


## Troubleshooting & next steps

**`RuntimeError: company search failed: HTTP 429`**
You exceeded **200 requests/minute**. The client already retries with backoff, so this means sustained overload — space out calls, or lower `max_companies` / `max_workers`.

**A company shows `decision_makers: []` — is that real?**
Check `decision_makers_complete`. `True` means genuinely nobody on-ICP was found (normal for showcase pages and tiny firms). `False` means a lookup was abandoned — check `companies[0]["_errors"]`.

**Too many loosely-related companies?** Raise `min_score` (try `0.5`), or pass both `url` and `name` so the domain can anchor the match.

**Missing someone you know is there?** Loosen `min_tier` to `4`, raise `max_contacts`, or add their title pattern to `ICP`. The title filter is deliberately strict — it targets finance and operations buyers.

**Need a field this index does not carry** (emails, phone numbers, tech stack, custom questions)? That is what [`enrich-company`](sixtyfour_enrich_company_tutorial.ipynb) is for — slower, but it can research anything you can describe. A common pattern is to use this notebook to resolve *which* company you mean, then hand the winning domain to `enrich-company` for depth.

**Remember**
- Rate limit: **200 requests/minute** for `/search/query`
- One call here costs roughly `2 + (companies × 6)` requests
- `email` and `phone` are always `None` — use `find-email` / `find-phone` to fill them